# Prepare imports and data

Downloading data using the `mlcroissant` library

In [71]:
import mlcroissant as mlc
import pandas as pd
import re # regex

# downloads the dataset
croissant_dataset = mlc.Dataset('https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis/croissant/download')

  -  [Metadata(Twitter Sentiment Analysis)] Property "http://mlcommons.org/croissant/citeAs" is recommended, but does not exist.


This is the raw data that is produced straight from the dataset

In [72]:
record_sets = croissant_dataset.metadata.record_sets
print(record_sets) # we have two datasets: "twitter_training" and "twitter_validation"

df = pd.DataFrame(croissant_dataset.records(record_set=record_sets[0].uuid))
df.head()

[RecordSet(uuid="twitter_training.csv"), RecordSet(uuid="twitter_validation.csv")]


,twitter_training.csv/2401,twitter_training.csv/Borderlands,twitter_training.csv/Positive,twitter_training.csv/im+getting+on+borderlands+and+i+will+murder+you+all+%2C
0,2401,b'Borderlands',b'Positive',b'I am coming to the borders and I will kill y...
1,2401,b'Borderlands',b'Positive',b'im getting on borderlands and i will kill yo...
2,2401,b'Borderlands',b'Positive',b'im coming on borderlands and i will murder y...
3,2401,b'Borderlands',b'Positive',b'im getting on borderlands 2 and i will murde...
4,2401,b'Borderlands',b'Positive',b'im getting into borderlands and i can murder...


## Initial data analysis

Right now, looks like we have 4 available columns:
- ID??/Class id??? (unsure, not required)
- Company/Genre/Username (again, unsure. the dataset doesn't give a lot of information)
- Sentiment, classified as either `Positive`, `Negative`, `Neutral` or `Irrelevant`. In this dataset, it was recommended that `Irrelevant = Neutral`. Each of the classifications are what they mean, no need to explain. 
- The text content itself. 

To use this model, we only need two pieces of data: 
- Sentiment
- Text content (tokenised)

The others are not required/idk what they do/they probably won't help that much

The text content is not in its correct format - it is currently in a word format. This cannot be analysed, so we must put it through a process called tokenisation. 

Of course, tokenisation is not possible without first cleaning up the data. 

# Quick cleanup of data

Some of the columns are not necessary at all. I will purge them and preppare for preprocessing

In [ ]:
df = df.drop(columns=['twitter_training.csv/2401', 'twitter_training.csv/Borderlands']) # remove unnecessary columns
df = df.drop(index=0).reset_index(drop=True) # first one is poorly formatted

# the original data formatting is poorly formatted, so lets format it properly. 
df.columns = ['Sentiment', 'Text Content']

# clean up the b'{}' stuff
df['Sentiment'] = df['Sentiment'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
df['Text Content'] = df['Text Content'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# randomise the data so we can check out something else other than the top 5. 
df = df.sample(frac=1).reset_index(drop=True)

df.head()

,Sentiment,Text Content
0,Negative,@ Afvision hey i am suffering from the problem...
1,Neutral,Microsoft Edge Is Sound More Popular Like Fire...
2,Neutral,AMD is supposedly testing Big Navi - and it co...
3,Positive,you @MagzOnYT and I have hit have a lot of fun...
4,Neutral,"This punk thought this was sneaky, he trapped ..."


# Preprocessing

## Imports

In this section, I am using NLTK (Natural Language Toolkit), which contains utilities for being able to convert sentences into tokens.   

In [74]:
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
import re
import contractions

In [75]:
nltk.download('stopwords')
nltk.download('twitter_samples')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package twitter_samples to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Stopwords

Stopwords are words that carry semantic meaning and can make analysis of the words hard. It includes such articles ("a", "the", ...), conjunctions ("and", "or", ...), prepositions ("in", "on", ...) and other such words. 

By removing stopwords, we are able to remove any tokens/words that do not hold minimal meaning. 

In [76]:
stop_words = set(stopwords.words('english'))

There are some words that can drastically change the meaning if removed. 

For example:

"That is not good" (negative) ----removing stopword---> "That is good" (positive) *[not what we want]*

By excluding the negations from the stopwords, we will be able to keep the sentiment of the text block

In [77]:
negations = {'no', 'not', 'nor', 'neither', 'never', 'none',
            "don't", "won't", "can't", "isn't", "aren't", "wasn't"}
stop_words = stop_words - negations

## Tokenising

By tokenising, we are able to convert a long block of text into individual words. 

In [78]:
tokeniser = TweetTokenizer(strip_handles=True, reduce_len=True)

In [79]:
def clean_text_content(text):
    if pd.isna(text):
        return ""
    text = str(text)

    text = re.sub(r'[^\x00-\x7F]+', '', text)     # remove non-ASCII
    text = re.sub(r'http\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'#\w+', '', text)              # remove hashtags
    text = re.sub(r'&\w+;', '', text)             # remove HTML entities
    text = contractions.fix(text)                 # expand contractions
    text = text.lower().strip()
    return text

df['Text Content'] = df['Text Content'].apply(clean_text_content)
df.head()

,Sentiment,Text Content
0,Negative,@ afvision hey i am suffering from the problem...
1,Neutral,microsoft edge is sound more popular like fire...
2,Neutral,amd is supposedly testing big navi - and it co...
3,Positive,you @magzonyt and i have hit have a lot of fun...
4,Neutral,"this punk thought this was sneaky, he trapped ..."


In [80]:
def tokenise_text_content(text):
    if not text:
        return []
    tokens = tokeniser.tokenize(text)
    tokens = [re.sub(r'[^\w\s]', '', w) for w in tokens]  # remove punctuation
    tokens = [w for w in tokens if w and w not in stop_words]
    return tokens

df["Text Tokens"] = df["Text Content"].apply(tokenise_text_content)
df.head()

,Sentiment,Text Content,Text Tokens
0,Negative,@ afvision hey i am suffering from the problem...,"[afvision, hey, suffering, problem, call, duty..."
1,Neutral,microsoft edge is sound more popular like fire...,"[microsoft, edge, sound, popular, like, firefo..."
2,Neutral,amd is supposedly testing big navi - and it co...,"[amd, supposedly, testing, big, navi, could, m..."
3,Positive,you @magzonyt and i have hit have a lot of fun...,"[hit, lot, fun, fortnite, recently, video]"
4,Neutral,"this punk thought this was sneaky, he trapped ...","[punk, thought, sneaky, trapped]"


## Classification of sentiment

As previously mentioned, there are ~~4~~ 3 primary types of sentiment classification: 
- Positive
- Neutral/Irrelevant
- Negative

Within the set, we can use multi-class classification as so:
- Positive (2)
- Negative (1) (it only made sense if neutral was 0, not Negative)
- Neutral/Irrelevant (0)

In [81]:
s_map = {
    'Positive': 2,
    'Negative': 1,
    'Neutral': 0,
    'Irrelevant': 0,
}

df['Sentiment'] = df['Sentiment'].map(s_map)
df.head()

,Sentiment,Text Content,Text Tokens
0,1,@ afvision hey i am suffering from the problem...,"[afvision, hey, suffering, problem, call, duty..."
1,0,microsoft edge is sound more popular like fire...,"[microsoft, edge, sound, popular, like, firefo..."
2,0,amd is supposedly testing big navi - and it co...,"[amd, supposedly, testing, big, navi, could, m..."
3,2,you @magzonyt and i have hit have a lot of fun...,"[hit, lot, fun, fortnite, recently, video]"
4,0,"this punk thought this was sneaky, he trapped ...","[punk, thought, sneaky, trapped]"
